# 01 — Dataset Preparation
## Explainable Multimodal Quantum-GNN Framework for Chest X-ray Diagnosis

This notebook:
1. Sets up the environment on Google Colab (GPU)
2. Loads and inspects the NIH ChestX-ray14 dataset
3. Generates statistics and visualizations
4. Creates patient-level train / val / test splits
5. Builds the patient graph
6. Runs the pipeline smoke test

**Dataset**: `chehablab/NIHChestXR` — 112,120 chest X-rays from 30,805 patients

**Runtime**: GPU (T4 recommended). CPU is OK for stats only.

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — running on CPU')

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
import sys

print('Installing core ML dependencies...')
!pip install -q datasets huggingface_hub timm einops rich tqdm
!pip install -q scikit-learn pandas matplotlib seaborn pyyaml omegaconf
!pip install -q grad-cam shap captum

# PyG (torch-geometric)
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'torch={torch_version}, cuda={cuda_version}')

!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html

# PennyLane
!pip install -q pennylane pennylane-lightning
print('All dependencies installed.')

In [ ]:
# ── 2. Clone / Mount project ─────────────────────────────────────────────
# Option A: Mount Google Drive (if project is in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/urop

# Option B: Run directly from Colab (no Drive)
import os
PROJECT_DIR = '/content/urop'

if not os.path.exists(PROJECT_DIR):
    # If you have a GitHub repo, clone it:
    # !git clone https://github.com/pchandraswaroop/UROP.git {PROJECT_DIR}
    # Otherwise, upload your project files and set PROJECT_DIR accordingly.
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print(f'Created {PROJECT_DIR} — upload your project files here.')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 3. Setup logging & reproducibility ───────────────────────────────────
from src.utils.logger import setup_logging
from src.utils.reproducibility import set_seed

setup_logging('INFO')
set_seed(42)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 4. Load dataset and inspect schema ───────────────────────────────────
from src.dataset.hf_loader import load_hf_dataset, inspect_schema, get_dataset_stats_metadata

CACHE_DIR = 'data/raw'

# Load small sample first for inspection
print('Loading 500-sample subset for inspection...')
ds_sample = load_hf_dataset(cache_dir=CACHE_DIR, sample_n=500)

print('\n=== SCHEMA ===')
schema = inspect_schema(ds_sample)
for col, info in schema.items():
    print(f'  {col}: {info}')

In [ ]:
# ── 5. Visualize sample images ────────────────────────────────────────────
import matplotlib.pyplot as plt
from src.dataset.label_encoder import NIH_LABEL_NAMES

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for i, ax in enumerate(axes.flat):
    if i >= len(ds_sample): break
    row = ds_sample[i]
    label_names = [NIH_LABEL_NAMES[j] for j in row['labels'] if j < len(NIH_LABEL_NAMES)]
    ax.imshow(row['image'], cmap='bone')
    ax.set_title(f"P:{row['patient_id']}\nAge:{row['age']} {row['sex']}\n{', '.join(label_names)}",
                 fontsize=8)
    ax.axis('off')

plt.suptitle('NIH ChestX-ray14 Sample Images', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('data/stats/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: data/stats/sample_images.png')

In [ ]:
# ── 6. Run full statistics analysis ───────────────────────────────────────
# For full dataset stats: remove --sample_n or set to -1
!python scripts/analyze_statistics.py \
    --cache_dir data/raw \
    --sample_n 5000 \
    --stats_dir data/stats

In [ ]:
# ── 7. Display generated figures ──────────────────────────────────────────
from IPython.display import Image as IPyImage, display
import os

figs = ['data/stats/label_distribution.png',
        'data/stats/age_distribution.png',
        'data/stats/sex_distribution.png',
        'data/stats/patient_scan_count.png',
        'data/stats/cooccurrence_matrix.png']

for fig_path in figs:
    if os.path.exists(fig_path):
        print(f'\n--- {fig_path} ---')
        display(IPyImage(fig_path))

In [ ]:
# ── 8. Load full dataset and build splits ────────────────────────────────
# WARNING: Full dataset download is ~2.5 GB. First run may take 10-30 min.
print('Loading full dataset (may take a while on first run)...')
!python scripts/build_splits.py \
    --cache_dir data/raw \
    --split_dir data/splits

# Verify splits
import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/splits/{split}.csv')
    print(f'{split}: {len(df):,} scans, {df["patient_id"].nunique():,} patients')

In [ ]:
# ── 9. Build patient graph ────────────────────────────────────────────────
!python scripts/build_graph.py \
    --split_dir data/splits \
    --graph_path data/patient_graph.pt \
    --edge_threshold 0.3 \
    --max_patients 1000  # Remove limit for full graph build

In [ ]:
# ── 10. Run pipeline smoke test ───────────────────────────────────────────
!python scripts/verify_pipeline.py --n_samples 64 --cache_dir data/raw

In [ ]:
# ── 11. Run unit tests ────────────────────────────────────────────────────
!pytest tests/ -v --tb=short

In [ ]:
# ── 12. Sample DataLoader ─────────────────────────────────────────────────
import torch
import pandas as pd
from src.dataset.hf_loader import load_hf_dataset
from src.dataset.dataloader import get_dataloader

# Use small subset for demonstration
ds = load_hf_dataset(cache_dir='data/raw', sample_n=200)
train_df = pd.read_csv('data/splits/train.csv')

# Filter to only rows in our sample
sample_hf_indices = set(range(200))
train_df_sample = train_df[train_df['hf_index'].isin(sample_hf_indices)].head(100)

BATCH_SIZE = 32 if DEVICE == 'cuda' else 4
loader = get_dataloader('train', train_df_sample, ds,
                         batch_size=BATCH_SIZE, num_workers=2)

print(f'DataLoader: {len(loader)} batches of size {BATCH_SIZE}')
batch = next(iter(loader))
print(f'  image shape:  {batch["image"].shape}')
print(f'  label shape:  {batch["label"].shape}')
print(f'  meta shape:   {batch["meta"].shape}')
print(f'  label sample: {batch["label"][0].nonzero().squeeze().tolist()}')

In [ ]:
# ── 13. GNN Limitation Summary ────────────────────────────────────────────
print("""
=== GNN DESIGN LIMITATIONS ===

Available non-label metadata: age, sex (2 features only)

GNN limitations:
  • Node features are weak (only age_norm + sex_binary)
  • Edges built from training labels → unavailable at test time
  • New patients have no graph connections (inductive failure)

Mitigations:
  • MLP baseline on metadata implemented for comparison
  • Disease-specific fusion gate down-weights weak clinical branch
  • GraphSAGE (inductive) used instead of GCN for better generalisation
  • Clinical branch automatically falls back to MLPBaseline for val/test

Recommendation:
  • Compare GNN vs MLP AUC after training
  • If GNN AUC ≤ MLP AUC + 1%, disable GNN branch in final model
""")

## ✅ Dataset Preparation Complete

**Deliverables generated:**
- `data/splits/train.csv` / `val.csv` / `test.csv` — patient-stratified splits
- `data/stats/dataset_report.md` — full statistics report
- `data/stats/*.png` — visualizations
- `data/patient_graph.pt` — patient-level PyG graph

**Next steps (Prompt 2):**
- Implement full training loop
- Train image branch baseline
- Compare MLP vs GNN clinical branch
- Implement explainability modules
- Build Streamlit frontend